# SIPTA Notebook: Ingestión de datos — Dominio Finanzas / Economía Informal (D6)
Este notebook documenta la lectura, consolidación temporal de vendedores informales y la carga de infraestructura productiva (Puntos de Encuentro IPES).

### Objetivos
- Consolidar las 6 series semestrales de **Vendedores Informales (RIVI)** de 2017 a 2019.
- Cargar y estructurar los **Puntos de Encuentro de Vendedores** desde Excel/GeoJSON.
- Validar esquemas, llaves territoriales de localidad y anomalías en coordenadas.
- Exportar las copias versionadas a `data/raw/FINANZAS/`.

In [ ]:
from pathlib import Path
import re
import logging
import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

ROOT = Path('..').resolve()
RAW_DIR = ROOT / 'data' / 'raw'
FINANZAS_DIR = RAW_DIR / 'FINANZAS'
FINANZAS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Directorio Raw Finanzas: {FINANZAS_DIR}")

## 1. Consolidación de Series: Vendedores Informales (RIVI)

In [ ]:
def consolidate_vendedores_informales(input_dir: Path) -> pd.DataFrame:
    """Lee los archivos txt de RIVI, extrae la fecha del nombre y estandariza columnas."""
    files = sorted(list(input_dir.glob("rivi-numero-vendedores-informales-localidad-*.txt")))
    if not files:
        files = sorted(list(RAW_DIR.glob("rivi-numero-vendedores-informales-localidad-*.txt")))
    
    assert len(files) > 0, f"No se encontraron archivos RIVI en {input_dir}"
    
    dfs = []
    for file in files:
        match = re.search(r'(\d{4}-\d{2}-\d{2})', file.name)
        fecha = match.group(1) if match else "Desconocida"
        
        try:
            df = pd.read_csv(file, sep='\t', encoding='utf-8')
        except UnicodeDecodeError:
            df = pd.read_csv(file, sep='\t', encoding='latin1')
            
        if 'IndiceRespuesta' in df.columns:
            df = df.rename(columns={
                'IndiceRespuesta': 'codigo_localidad',
                'NumeroLocalidad': 'nombre_localidad',
                'Numero': 'numero_vendedores',
                'Porcentaje': 'porcentaje'
            })
        elif 'NombreLocalidad' in df.columns:
            df = df.rename(columns={
                'NumeroLocalidad': 'codigo_localidad',
                'NombreLocalidad': 'nombre_localidad',
                'Numero': 'numero_vendedores',
                'Porcentaje': 'porcentaje'
            })
            
        df_clean = df[['codigo_localidad', 'nombre_localidad', 'numero_vendedores', 'porcentaje']].copy()
        df_clean['fecha_corte'] = fecha
        df_clean['archivo_origen'] = file.name
        dfs.append(df_clean)
        
    df_final = pd.concat(dfs, ignore_index=True)
    return df_final

df_vendedores = consolidate_vendedores_informales(FINANZAS_DIR)
print(f"✓ Vendedores Informales consolidados: {len(df_vendedores)} filas.")
display(df_vendedores.head())

## 2. Ingesta: Puntos de Encuentro de Vendedores (IPES)

In [ ]:
excel_path = FINANZAS_DIR / 'Punto de encuentro vendedores. Bogotá D.C..xlsx'
if not excel_path.exists():
    excel_path = RAW_DIR / 'Punto de encuentro vendedores. Bogotá D.C..xlsx'

df_puntos = pd.read_excel(excel_path)
print(f"✓ Puntos de encuentro cargados: {len(df_puntos)} registros.")

cols_puntos = [
    'properties/CPUNNOM',
    'properties/CPUNDIR',
    'properties/CPUNLOC',
    'properties/CPUNBARRIO',
    'properties/CPUNHORAPV',
    'properties/CPUNNUMLOC'
]
display(df_puntos[cols_puntos].head())

In [ ]:
print("=== REVISIÓN DE LLAVES Y COBERTURA ===")
print("Fechas RIVI:", sorted(df_vendedores['fecha_corte'].unique()))
print("Localidades en Puntos de Encuentro:", df_puntos['properties/CPUNLOC'].unique().tolist())

# Guardar copias consolidadas en raw
df_vendedores.to_csv(FINANZAS_DIR / "vendedores_informales_consolidado.csv", index=False, encoding='utf-8')
df_puntos.to_csv(FINANZAS_DIR / "puntos_encuentro_vendedores.csv", index=False, encoding='utf-8')
print("✓ Respaldos generados en data/raw/FINANZAS/")

### Notas de Ingesta — Finanzas / Economía (D6)
1. **Vendedores Informales (RIVI):**
   - 126 registros semestrales (2017-06-30 a 2019-12-30).
   - Llave territorial: `codigo_localidad` (1 a 20 y código 160).
2. **Puntos de Encuentro (IPES):**
   - 4 equipamientos comerciales regulados de apoyo al vendedor informal (Alcalá, Aguas, Mundo Aventura y Tintal).
   - **⚠ Pendiente de validar con datos (Inconsistencia en código de localidad):** La columna `CPUNNUMLOC` registra '18' para Alcalá (Usaquén) y '12' para Aguas (Santa Fe); se corregirá mapeando con `CPUNLOC` en la fase 1.4 de limpieza.
   - **⚠ Pendiente de validar con datos (Escala decimal de coordenadas):** Las columnas `geometry/coordinates` en Excel perdieron el punto decimal (`-7405083...`); se normalizarán a coordenadas geográficas WGS84 en la limpieza.